In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp /content/drive/MyDrive/LanguageModel/61262-0.txt .

In [40]:
# CONFIGURATION AND HYPERPARAMETER SETUP
# ------------------------------------------------------------
# Defines training parameters, model architecture settings,
# vocabulary constraints and reproducibility seeds.

import os, re, json, math, random
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

TEXT_PATH = "61262-0.txt"
OUT_DIR = "/content/drive/MyDrive/wordlm_submission"
os.makedirs(OUT_DIR, exist_ok=True)

# Data / training
SEQ_LEN = 50
BATCH_SIZE = 64
EPOCHS = 25
TRAIN_FRAC = 0.9

# Vocabulary
MIN_FREQ = 1
MAX_VOCAB = 12000
ADD_EOS = True

# Model
EMB_DIM = 128
RNN_UNITS = 256
NUM_LAYERS = 1
DROPOUT = 0.4
LR = 3e-4

In [39]:
# TEXT LOADING, NORMALISATION AND TOKENISATION
# ------------------------------------------------------------
#  Removes Project Gutenberg metadata
#  Normalises punctuation (curly quotes, dashes, etc)
#  Tokenises text into word level tokens
#  Optionally inserts <eos> tokens after sentence endings
#  Builds vocabulary with frequency threshold and cap
#  Encodes tokens into integer IDs

from collections import Counter

TOKEN_PATTERN = r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[.,!?;:()\"-]"
SPECIAL_TOKENS = ["<pad>", "<unk>", "<eos>"]

def read_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def strip_gutenberg(text: str) -> str:
    start_markers = [
        "*** START OF THIS PROJECT GUTENBERG EBOOK",
        "*** START OF THE PROJECT GUTENBERG EBOOK",
        "***START OF THIS PROJECT GUTENBERG EBOOK",
    ]
    end_markers = [
        "*** END OF THIS PROJECT GUTENBERG EBOOK",
        "*** END OF THE PROJECT GUTENBERG EBOOK",
        "***END OF THIS PROJECT GUTENBERG EBOOK",
    ]

    start_idx = None
    for m in start_markers:
        i = text.find(m)
        if i != -1:
            start_idx = i
            break
    if start_idx is not None:
        text = text[start_idx:]
        lines = text.splitlines()
        text = "\n".join(lines[2:])

    end_idx = None
    for m in end_markers:
        i = text.find(m)
        if i != -1:
            end_idx = i
            break
    if end_idx is not None:
        text = text[:end_idx]

    return text

def normalise_text(text: str) -> str:
    """
    Fix curly quotes/apostrophes/dashes that break tokenisation.
    This is a big quality win for your model.
    """
    replacements = {
        "\u2019": "'",
        "\u2018": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2014": "-",
        "\u2013": "-",
        "\xa0": " ",
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    text = text.replace("\r", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text: str):
    tokens = re.findall(TOKEN_PATTERN, text)
    if ADD_EOS:
        out = []
        for t in tokens:
            out.append(t)
            if t in [".", "!", "?"]:
                out.append("<eos>")
        return out
    return tokens

def build_vocab(tokens, min_freq=1, max_vocab=12000):
    counts = Counter(tokens)

    # Start with specials
    vocab = ["<pad>", "<unk>", "<eos>"] if ADD_EOS else ["<pad>", "<unk>"]

    # Add frequent tokens up to cap
    for w, c in counts.most_common():
        if w in vocab:
            continue
        if c < min_freq:
            continue
        vocab.append(w)
        if max_vocab is not None and len(vocab) >= max_vocab:
            break

    word_to_id = {w: i for i, w in enumerate(vocab)}
    id_to_word = {i: w for w, i in word_to_id.items()}
    return word_to_id, id_to_word, counts

def encode(tokens, word_to_id):
    unk = word_to_id["<unk>"]
    return np.array([word_to_id.get(t, unk) for t in tokens], dtype=np.int32)

raw = read_text(TEXT_PATH)
cleaned = strip_gutenberg(raw)
cleaned = normalise_text(cleaned)

tokens = tokenize(cleaned)
word_to_id, id_to_word, counts = build_vocab(tokens, min_freq=MIN_FREQ, max_vocab=MAX_VOCAB)

vocab_size = len(word_to_id)
pad_id = word_to_id["<pad>"]
unk_id = word_to_id["<unk>"]
eos_id = word_to_id.get("<eos>", None)

ids = encode(tokens, word_to_id)

print("Total tokens:", len(tokens))
print("Vocab size:", vocab_size)
print("Top 15 tokens:", counts.most_common(15))
print("Example tokens:", tokens[:40])

# Save vocab
with open(os.path.join(OUT_DIR, "vocab.json"), "w", encoding="utf-8") as f:
    json.dump(
        {
            "word_to_id": word_to_id,
            "id_to_word": {str(k): v for k, v in id_to_word.items()},
            "min_freq": MIN_FREQ,
            "max_vocab": MAX_VOCAB,
            "seq_len": SEQ_LEN,
            "add_eos": ADD_EOS
        },
        f,
        indent=2
    )

print("Saved vocab to:", os.path.join(OUT_DIR, "vocab.json"))

Total tokens: 71819
Vocab size: 6530
Top 15 tokens: [('<eos>', 5218), ('.', 4148), (',', 3686), ('"', 3474), ('the', 2713), ('to', 1330), ('I', 1288), ('a', 1237), ('of', 1222), ('and', 1090), ('-', 805), ('was', 735), ('in', 734), ('?', 656), ('that', 631)]
Example tokens: ['POIROT', 'INVESTIGATES', 'BY', 'THE', 'SAME', 'AUTHOR', 'THE', 'MYSTERIOUS', 'AFFAIR', 'AT', 'STYLES', 'THE', 'SECRET', 'ADVERSARY', 'THE', 'MURDER', 'ON', 'THE', 'LINKS', 'THE', 'BODLEY', 'HEAD', 'POIROT', 'INVESTIGATES', 'BY', 'AGATHA', 'CHRISTIE', 'LONDON', 'JOHN', 'LANE', 'THE', 'BODLEY', 'HEAD', 'LIMITED', 'First', 'published', 'in', 'Great', 'Britain', 'by']
Saved vocab to: /content/drive/MyDrive/wordlm_submission/vocab.json


In [29]:
# TRAIN / VALIDATION SPLIT AND DATASET CREATION
# ------------------------------------------------------------
# Splits encoded token IDs into training and validaiton sets
# Creates sliding window sequences of length SEQ_LEN
# Constructs tf.data pipelines with batching and prefetching
# Prepares input-target pairs for next word prediction

split = int(len(ids) * TRAIN_FRAC)
train_ids = ids[:split]
val_ids = ids[split:]

print("Train tokens:", len(train_ids))
print("Val tokens:", len(val_ids))

def make_dataset(token_ids: np.ndarray, seq_len: int, batch_size: int, shuffle: bool):
    ds = tf.data.Dataset.from_tensor_slices(token_ids)
    ds = ds.window(seq_len + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda w: w.batch(seq_len + 1))

    def split_xy(chunk):
        x = chunk[:-1]
        y = chunk[-1]
        return x, y

    ds = ds.map(split_xy, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(20000, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
    return ds

Train tokens: 64637
Val tokens: 7182


In [30]:
# MODEL ARCHITECTURE (WORD LEVEL LSTM LANGUAGE MODEL)
# ------------------------------------------------------------
# Embedding layer for token representation
# LSTM recurrent layers for sequence modelling
# Layer normalisation and dropout for stability
# Output dense layer to vocabulary logits
# Compiles model with cross entropy loss

def build_lstm_lm(vocab_size, emb_dim, rnn_units, num_layers, dropout):
    inputs = tf.keras.Input(shape=(SEQ_LEN,), dtype=tf.int32)

    x = tf.keras.layers.Embedding(vocab_size, emb_dim)(inputs)

    # LSTM
    for i in range(num_layers):
        return_seq = (i < num_layers - 1)
        x = tf.keras.layers.LSTM(
            rnn_units,
            return_sequences=return_seq,
            dropout=dropout,
            recurrent_dropout=0.0,
            kernel_regularizer=tf.keras.regularizers.l2(1e-6)
        )(x)

    x = tf.keras.layers.LayerNormalization()(x)
    x = tf.keras.layers.Dropout(dropout)(x)

    logits = tf.keras.layers.Dense(
        vocab_size,
        kernel_regularizer=tf.keras.regularizers.l2(1e-6)
    )(x)

    return tf.keras.Model(inputs, logits)

model = build_lstm_lm(vocab_size, EMB_DIM, RNN_UNITS, NUM_LAYERS, DROPOUT)
model.summary()

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR, clipnorm=1.0),
    loss=loss_fn,
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_3 (Embedding)         │ (None, 50, 128)        │       835,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_1           │ (None, 256)            │           512 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6530)           │     1,678,210 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,908,802 (11.10 MB)

 Trainable params: 2,908,802 (11.10 MB)

 Non-trainable params: 0 (0.00 B)

In [41]:
# MODEL TRAINING WITH CHECKPOINT AND EARLY STOPPING
# ------------------------------------------------------------
# Builds finite training and validation datasets
# Computes steps_per_epoch to avoid infinite iteration
# Uses ModelCheckpoint to save best weights
# Uses ReduceLROnPlateau for adaptive learning rate
# Uses EarlyStopping to prevent overfitting
# Saves configuration for reproducibility

import os, json

train_ds_raw = make_dataset(train_ids, SEQ_LEN, BATCH_SIZE, shuffle=True)
val_ds_raw   = make_dataset(val_ids,   SEQ_LEN, BATCH_SIZE, shuffle=False)

steps_per_epoch = max(1, (len(train_ids) - SEQ_LEN) // BATCH_SIZE)
val_steps       = max(1, (len(val_ids)   - SEQ_LEN) // BATCH_SIZE)

print("steps_per_epoch:", steps_per_epoch)
print("val_steps:", val_steps)

train_ds = train_ds_raw.repeat().take(steps_per_epoch)
val_ds   = val_ds_raw.take(val_steps)

for xb, yb in train_ds.take(1):
    print("x batch shape:", xb.shape)
    print("y batch shape:", yb.shape)

ckpt_path = os.path.join(OUT_DIR, "best.weights.h5")

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        ckpt_path,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-5,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

print("Saved best weights to:", ckpt_path)

config = dict(
    seq_len=SEQ_LEN,
    batch_size=BATCH_SIZE,
    emb_dim=EMB_DIM,
    rnn_units=RNN_UNITS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    lr=LR,
    epochs_requested=EPOCHS,
    epochs_trained=len(history.history["loss"]),
    min_freq=MIN_FREQ,
    max_vocab=MAX_VOCAB,
    add_eos=ADD_EOS,
    train_frac=TRAIN_FRAC
)

config_path = os.path.join(OUT_DIR, "config.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Saved config to:", config_path)

steps_per_epoch: 1009
val_steps: 111
x batch shape: (64, 50)
y batch shape: (64,)
Epoch 1/25
1007/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - acc: 0.2929 - loss: 4.0273
Epoch 1: val_loss improved from inf to 5.19752, saving model to /content/drive/MyDrive/wordlm_submission/best.weights.h5
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 24s 18ms/step - acc: 0.2930 - loss: 4.0276 - val_acc: 0.2383 - val_loss: 5.1975 - learning_rate: 3.0000e-04
Epoch 2/25
1004/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - acc: 0.3068 - loss: 4.0255
Epoch 2: val_loss did not improve from 5.19752

Epoch 2: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 22s 19ms/step - acc: 0.3068 - loss: 4.0256 - val_acc: 0.2376 - val_loss: 5.2287 - learning_rate: 3.0000e-04
Epoch 3/25
1004/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - acc: 0.3234 - loss: 3.8389
Epoch 3: val_loss did not improve from 5.19752

Epoch 3: ReduceLROnPlateau reducing learning rate to 7.500000356230885e-05.
1009/1009 ━━━━━━━━━

In [35]:
# TEXT GENERATION (INFERANCE)
# ------------------------------------------------------------
# Loads best trained weights
# Implements nucleus (top-p) sampling
# Controls temperature
# Limits consecutive punctuation tokens
# Stops generation at <eos> token
# Generates similar text for given a prompt

import numpy as np

model.load_weights(os.path.join(OUT_DIR, "best.weights.h5"))
print("Loaded best weights.")

PUNCT_TOKENS = {".", ",", "!", "?", ";", ":", "\"", "-", ")"}

def join_tokens(tokens_list):
    no_space_before = {".", ",", "!", "?", ";", ":", ")", "\""}
    no_space_after = {"(", "\""}
    out = []
    for t in tokens_list:
        if t == "<eos>":
            continue
        if not out:
            out.append(t)
            continue
        if t in no_space_before:
            out[-1] = out[-1] + t
        elif out[-1] in no_space_after:
            out[-1] = out[-1] + t
        else:
            out.append(" " + t)
    return "".join(out)

def encode_prompt(prompt_text: str):
    prompt_text = normalise_text(prompt_text)
    prompt_tokens = re.findall(TOKEN_PATTERN, prompt_text.strip())
    prompt_ids = [word_to_id.get(t, unk_id) for t in prompt_tokens]
    return prompt_tokens, prompt_ids

def nucleus_sample(logits, temperature=0.75, top_p=0.9, banned_ids=None):
    logits = logits / max(1e-8, temperature)
    probs = tf.nn.softmax(logits).numpy()

    if banned_ids:
        probs = probs.copy()
        probs[banned_ids] = 0.0
        total = probs.sum()
        if total > 0:
            probs /= total

    sorted_idx = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]
    cumsum = np.cumsum(sorted_probs)

    cutoff = np.searchsorted(cumsum, top_p) + 1
    keep = sorted_idx[:cutoff]
    keep_probs = probs[keep]
    keep_probs /= keep_probs.sum()

    return int(np.random.choice(keep, p=keep_probs))

def generate_text(
    prompt_text,
    max_new_tokens=160,
    temperature=0.75,
    top_p=0.9,
    min_stop_len=20,
    max_consecutive_punct=1
):
    prompt_tokens, prompt_ids = encode_prompt(prompt_text)

    context = prompt_ids[-SEQ_LEN:]
    if len(context) < SEQ_LEN:
        context = [pad_id] * (SEQ_LEN - len(context)) + context

    x = tf.constant([context], dtype=tf.int32)
    generated_ids = []

    banned_base = [pad_id, unk_id]

    consecutive_punct = 0

    for i in range(max_new_tokens):
        logits = model(x, training=False)[0]

        banned = list(banned_base)

        if generated_ids:
            last_word = id_to_word[generated_ids[-1]]

            if last_word in PUNCT_TOKENS:
                consecutive_punct += 1
            else:
                consecutive_punct = 0

            if consecutive_punct >= max_consecutive_punct:
                for p in PUNCT_TOKENS:
                    if p in word_to_id:
                        banned.append(word_to_id[p])

        next_id = nucleus_sample(
            logits,
            temperature=temperature,
            top_p=top_p,
            banned_ids=banned
        )

        generated_ids.append(next_id)
        x = tf.concat([x[:, 1:], tf.constant([[next_id]], dtype=tf.int32)], axis=1)

        if eos_id is not None and next_id == eos_id and i >= min_stop_len:
            break

    out_tokens = [id_to_word[i] for i in generated_ids]
    return join_tokens(prompt_tokens + out_tokens)

# Testing using prompt
for p in ["Poirot looked at me and said", "Hastings, you must understand"]:
    print("\nPROMPT:", p)
    print(generate_text(p))

Loaded best weights.

PROMPT: Poirot looked at me and said
Poirot looked at me and said." What will be one of it?" said Poirot." A little great mistake.

PROMPT: Hastings, you must understand
Hastings, you must understand that the key to be to the side - is and that I should a sharp man - paper - and the murderer.


In [36]:
# INTERACTIVE PROMPT BASED TEXT GENERATION
# ------------------------------------------------------------
# Allows the user to type a prompt and generates a continuation
# using the trained language model. Type 'quit' to exit.

model.load_weights(os.path.join(OUT_DIR, "best.weights.h5"))
print("Loaded best weights. Type a prompt below (type 'quit' to stop).\n")

while True:
    prompt = input("Prompt> ").strip()
    if not prompt:
        continue
    if prompt.lower() == "quit":
        print("Exiting interactive generation.")
        break

    print("\nGenerated text:\n")
    print(generate_text(
        prompt_text=prompt,
        max_new_tokens=180,
        temperature=0.75,
        top_p=0.9,
        min_stop_len=20,
        max_consecutive_punct=1
    ))
    print("\n" + "-"*60 + "\n")

Loaded best weights. Type a prompt below (type 'quit' to stop).

Prompt> Poirot looked at me and said

Generated text:

Poirot looked at me and said, and then in the own next - room, and the letter he should go back to the lift.

------------------------------------------------------------

Prompt> Hastings, you must understand

Generated text:

Hastings, you must understand that the world is quite always said. I should asked myself in the road, and I know you that you have have gone to find my word.

------------------------------------------------------------

Prompt> quit
Exiting interactive generation.
